# 03b. 네 가지 흐름 ②: 분기와 루프, 그리고 패턴 번역

앞선 `03a`에서는 순차와 병렬 흐름을 구현했습니다. 이 두 흐름은 **업무의 선행 조건만으로** 실행 경로가 결정되었습니다. 이번 **`03b`** 노트북에서는 실행 중간의 상태 값을 확인하여 다음 단계를 결정하는 **분기(Branch)** 와 **루프(Loop)** 흐름을 다루고, 마지막으로 자주 쓰이는 멀티에이전트 패턴 이름들을 이 네 가지 기본 흐름으로 번역해 봅니다.

## 학습 목표

- 분기와 루프 흐름을 `StateGraph`로 직접 구현하고, 각 흐름에서 발생할 수 있는 대표적인 실패 사례를 재현합니다.
- Router의 단순 정확도 수치 하나가 아니라, 오분류 방향에 따른 실제 비용(누락 위험 vs 추가 호출 비용)으로 분기를 평가합니다.
- 재시도 루프에는 최대 횟수 상한뿐만 아니라 '동일한 실패의 반복'을 감지하는 판정 조건이 함께 있어야 하는 이유를 실행으로 확인합니다.
- Router, Supervisor, Orchestrator-Worker, Handoff, Hierarchical 패턴을 네 가지 기본 흐름(순차·병렬·분기·루프)의 조합으로 번역합니다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 4 | 분기와 오분류 비용 | 8 |
| 5 | 루프와 종료 조건 | 9 |
| 6 | 패턴을 네 흐름으로 번역 | 7 |
| 7 | 직접 해보기와 정리 | 6 |

## 호출 규모

이 노트북은 **실제 모델**만 호출하여 실행합니다. `Day03_new/.env` 파일에 `ANTHROPIC_API_KEY`가 설정되어 있지 않으면 첫 번째 코드 셀에서 안내 메시지와 함께 실행이 멈춥니다. 모델을 호출하는 유료 셀은 실행 직전에 호출 규모를 한 줄로 먼저 고지하며, 비용은 설정된 본인 API 키로 청구됩니다.

그래프의 구조 확인, 규칙 기반 Router, 재시도 시나리오 시뮬레이션처럼 모델의 추론이 필요 없는 부분은 일반 Python 함수 노드로 실행합니다. 이러한 셀들은 대부분 출력 마지막에 `모델 호출 0회`를 표시합니다. 환경 준비 셀, 그래프를 컴파일하거나 시각화하기만 하는 셀, 앞선 실행 기록을 표로 출력하기만 하는 셀 역시 별도 표시가 없더라도 모델을 호출하지 않습니다.

| 구분 | 유료 셀 수 | 모델 호출(추정) |
|---|---:|---:|
| 모델 라우터 | 1 | 6 |
| 실제 루프 | 1 | 4~6 |
| Orchestrator-Worker 조립 | 1 | 8~15 |
| 기본 실행 합계 | 3 | 약 18~27 |

실제 모델 호출 횟수는 실행할 때마다 조금씩 달라질 수 있습니다. 모델이 도구를 몇 번 호출할지는 실행 시점에 모델 스스로 결정하기 때문입니다.

아래 코드는 실습에 필요한 라이브러리를 불러오고 API 키 설정과 관통 사례(`T-1001`) 데이터를 확인합니다.

In [ ]:
# 기본 라이브러리 및 타입 힌트 불러오기
import json
import operator
import time
from typing import Annotated, Literal, TypedDict

# Deep Agents, LangChain, LangGraph 핵심 구성 요소 불러오기
from deepagents import create_deep_agent
from langchain.agents.structured_output import ProviderStrategy
from langchain_core.runnables import RunnableConfig
from langgraph.errors import GraphRecursionError, InvalidUpdateError
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, Send
from pydantic import BaseModel

# 실습용 계약(Contract), 작업대(Desk), 추적(Trace), 시각화 헬퍼 불러오기
from mas_lab.contracts import FinalAnswer, Route, WorkerResult, WorkerTask, join_results, merge_results, plan_hash, validate_final
from mas_lab.desk import Desk, load_billing, load_outage
from mas_lab.hybrid import ROLE_GOALS, ROLES
from mas_lab.runner import make_desk
from mas_lab.structures import OUTCOME_GUIDE, gate_answer, scope_line
from mas_lab.trace import ModelMeter, Trace, longest_path, overlap_seconds, timeline
from mas_lab.view import flows_of, show_graph, show_table, write_design_log
from mas_lab.workers import DENY_WRITES, ROLE_TOOLS, allowlist_middleware, contract_prompt, run_worker
from rag_lab.corpus import load_cases
from rag_lab.evidence import BudgetExceeded
from rag_lab.settings import announce, key_status, make_model, require_key

# API 키 확인 및 모델 초기화
settings = require_key()
model = make_model()

# 사례 데이터 로드 및 관통 사례(T-1001) 작업대 준비
CASES = {case["id"]: case for case in load_cases()}
T1001 = CASES["complex-double-pay"]
desk: Desk = make_desk(T1001, label="점검")

print(key_status(settings))
print("고객 요청 T-1001:", T1001["question"])
print("고객:", desk.scope.entity, "/ 기준일:", desk.scope.as_of, "/ 결제 건수:", len(desk.payments()))
print("측정 장치:", Trace.__name__, ModelMeter.__name__)

## 03a에서 이어 쓰는 값

4~6절에서는 앞선 `03a` 노트북에서 정의했던 변수와 헬퍼 함수 몇 가지를 이어서 사용합니다. 이 노트북은 `03a`의 커널 상태 없이 단독으로도 실행될 수 있어야 하므로, 필요한 값들만 아래 셀에서 다시 정의합니다.

| 값 | 03a에서 정의한 위치 | 이 노트북에서 사용하는 곳 |
|---|---|---|
| `CREDIT_PERCENT` | 2절의 순차 그래프 | 4절 Fast Path 실패 재현의 크레딧 금액 계산 |
| `walk` | 2절의 실패 재현 | 4절에서 그래프의 실행 경로 추적 및 상태 확인 |
| `PLAN` | 3절의 실제 Worker 실행 | 6절의 Orchestrator-Worker 그래프 조립 |
| `gold` | 3절의 필수 사실 검사 | 4절의 Router 비교표와 자동 검사, 6절의 최종 답안 자동 검사 |

새로 등장하는 개념은 없으므로, `03a`를 이미 진행했다면 가볍게 확인하고 넘어가시면 됩니다. 아래 코드는 이어서 사용할 상수와 헬퍼 함수, 평가용 기준 데이터를 준비합니다.

In [ ]:
# 03a에서 정의한 값 가운데 이 노트북에서 사용하는 항목만 다시 정의합니다.
CREDIT_PERCENT = 15  # data/business/sla-a-v2.md의 99.0% 이상 99.5% 미만 구간 크레딧 비율(%)

def walk(app, initial: dict, **config) -> tuple[list[str], dict]:
    """그래프의 실행 경로(Node 이름 리스트)와 최종 상태를 반환합니다. stream의 키가 곧 실행된 Node입니다."""
    path, state = [], dict(initial)
    for update in app.stream(initial, stream_mode="updates", config=config or None):
        for node, change in update.items():
            path.append(node)
            state.update(change or {})
    return path, state

# T-1001 사례에 대한 3개 역할(billing, outage, policy)의 작업 계획 정의
PLAN = {"case_id": T1001["id"], "question": T1001["question"],
        "tasks": [{"task_id": role, "goal": f"{ROLE_GOALS[role]}\n고객 요청: {T1001['question']}", "allowed_tools": sorted(ROLE_TOOLS[role])} for role in ROLES]}

# 사후 자동 검사용 평가 함수와 기준 데이터(gold) 로드
from mas_lab.evaluation import evaluate_row, load_gold, routing_table

gold = load_gold()

print("준비 완료 / 모델 호출 0회")

## 4절. 분기와 오분류 비용

**문제.** 모든 고객 요청에 동일한 고비용 파이프라인을 실행하지 않으려면 요청의 성격에 따라 경로를 나누어야 합니다. 그런데 경로를 분류하는 Router가 잘못 판단하면 어떤 비용이 발생할까요?

`Router`라는 이름이 거창해 보이지만 그래프의 실행 구조로 보면 **조건부 간선(Conditional Edge)** 하나입니다. 앞선 `03a`에서 `verify` 노드 뒤에 조건부 간선을 붙였던 것과 구조적으로 동일합니다. 달라지는 점은 '누가 그 조건을 판단하느냐'뿐입니다. 키워드 규칙 함수가 판단할 수도 있고, LLM 모델이 판단할 수도 있습니다. 어느 쪽이든 그래프 입장에서는 라우팅 함수가 다음 Node의 이름을 반환받아 이동할 뿐입니다.

| 경로 | 의미 | 비용 |
|---|---|---|
| Fast Path | 규정 한 곳이나 결제 한 건만 확인하면 되는 단순 요청입니다 | 낮음 |
| Slow Path | 결제, 규정, 장애 기록 중 두 가지 이상을 종합해야 하는 복합 요청입니다 | 높음 |
| 범위 밖 (`out_of_scope`) | 현재 고객이 아닌 다른 고객의 계약을 묻는 요청입니다 | 없음 (조사하지 않고 즉시 종료합니다) |

**먼저 예측하세요.** 규칙 기반 Router는 질문에 포함된 키워드 종류의 개수를 셉니다. `"같은 청구서를 두 번 결제하면 어떻게 처리되나요?"`라는 질문은 규정 문서 하나만 보면 되는 단순 질문이지만, 키워드 규칙은 이 질문을 어느 경로로 보낼까요? 반대로 모델 기반 Router는 이 문장을 어디로 분류할까요? 두 Router의 판단이 어긋날 만한 질문을 하나 더 골라 예측해 보세요.

아래 코드에서는 먼저 키워드 규칙으로 분기하는 `StateGraph`를 만들고, 샘플 질문 4개가 각각 어떤 경로로 이동하는지 확인합니다.

In [ ]:
# 질문에 포함된 키워드로 필요한 자료 종류를 판별하는 규칙 정의
ROUTE_WORDS = {"결제": ["결제", "환불", "청구"], "장애": ["장애", "가용률", "보상"], "규정": ["규정", "기한", "처리"]}
OTHER_CUSTOMERS = ["베타"]

def needed_kinds(question: str) -> list[str]:
    """질문에 등장하는 키워드를 기준으로 필요한 자료 분야 목록을 반환합니다."""
    return [kind for kind, words in ROUTE_WORDS.items() if any(word in question for word in words)]

def rule_route(question: str) -> str:
    """타 고객 언급 시 out_of_scope, 자료 종류가 2개 이상이면 slow, 그 외에는 fast를 반환합니다."""
    if any(name in question for name in OTHER_CUSTOMERS):
        return "out_of_scope"
    return "slow" if len(needed_kinds(question)) >= 2 else "fast"

# 라우팅 상태 정의 및 규칙 기반 분기 그래프 구성
class RouteState(TypedDict, total=False):
    question: str
    route: str

builder = StateGraph(RouteState)
builder.add_node("classify", lambda state: {"route": rule_route(state["question"])})
for name in ["fast", "slow", "out_of_scope"]:
    builder.add_node(name, lambda state: {})
    builder.add_edge(name, END)
builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", lambda state: state["route"], ["fast", "slow", "out_of_scope"])
rule_app = builder.compile()

# 규칙 조건표와 샘플 사례 4개의 라우팅 경로 출력
show_table([{"조건": "다른 고객 이름이 있다", "다음 Node": "out_of_scope"}, {"조건": "자료 종류가 둘 이상이다", "다음 Node": "slow"}, {"조건": "그 밖", "다음 Node": "fast"}])
samples = ["simple-deadline", "simple-duplicate-rule", "complex-double-pay", "scope-beta-from-alpha"]
show_table([{"사례": c, "질문": CASES[c]["question"], "자료 종류": needed_kinds(CASES[c]["question"]), "경로": " → ".join(walk(rule_app, {"question": CASES[c]["question"]})[0])} for c in samples])
print("모델 호출 0회")

In [ ]:
# 모델 기반 Router로 6개 사례를 분류하고 규칙 기반 Router 결과와 나란히 비교합니다.
announce("모델 라우터가 사례 6개를 Route로 분류합니다", model_calls=6)
ROUTER_CASES = ["simple-deadline", "simple-duplicate-rule", "complex-double-pay", "complex-ops-log", "insufficient-no-policy", "scope-beta-from-alpha"]
router_model = model.with_structured_output(Route, method="json_schema")

def model_route(case: dict) -> str:
    """구조화된 출력(Route)을 사용해 모델이 요청 경로(fast/slow/out_of_scope)를 분류합니다."""
    prompt = ("고객 요청의 경로를 고르세요. fast: 규정 한 곳이나 결제 한 건만 보면 되는 질문. slow: 결제, 규정, 장애 중 둘 이상이 필요한 질문. "
              f"out_of_scope: 현재 고객이 아닌 다른 고객의 계약을 묻는 질문. {scope_line(make_desk(case))}\n질문: {case['question']}")
    try:
        return router_model.invoke(prompt).path
    except Exception as error:  # 호출 실패도 숨기지 않고 표의 행으로 남깁니다.
        return f"오류:{type(error).__name__}"

# 6개 사례에 대해 모델 라우팅을 실행하고 기대 경로 및 규칙 경로와 비교표 출력
predicted = {case_id: model_route(CASES[case_id]) for case_id in ROUTER_CASES}
table = routing_table(predicted, gold)
show_table([{**row, "규칙 경로": rule_route(CASES[row["사례"]]["question"])} for row in table], ["사례", "기대", "예측", "일치", "규칙 경로"])
print(f"모델 라우터 일치: {sum(row['일치'] for row in table)}/{len(table)} (1회 관찰)")

In [ ]:
# 실패 재현: 복합 요청을 Fast Path로 강제로 보내 수집한 자료를 자동 검사합니다
SLOW_QUERIES = ["중복 결제 환불 규정", "월간 가용률 서비스 크레딧 비율", "서비스 크레딧 신청 기한"]

def collect_fast(case: dict) -> dict:
    """Fast Path: 고객 질문 원문 그대로 문서만 1회 검색합니다."""
    return {"규정": " ".join(e["quote"] for e in make_desk(case).context.search(case["question"], limit=3)["evidence"])}

def collect_slow(case: dict) -> dict:
    """Slow Path: 결제 내역, 가용률 기록, 세부 규정들을 모두 모으고 코드로 크레딧 금액을 계산합니다."""
    desk = make_desk(case)
    payments = [p for p in desk.payments() if p["invoice_id"] == "INV-2026-08"]
    quotes = [e["quote"] for query in SLOW_QUERIES for e in desk.context.search(query, limit=2)["evidence"]]
    return {"결제": json.dumps(payments, ensure_ascii=False), "장애": json.dumps(load_outage(case["entity"]), ensure_ascii=False),
            "규정": " ".join(quotes), "계산": f"크레딧 {payments[0]['base_fee_part'] * CREDIT_PERCENT // 100} 비율 {CREDIT_PERCENT}%"}

def check_collected(label: str, case: dict, collected: dict) -> dict:
    """수집된 자료 텍스트 안에 필수 사실들이 포함되어 있는지 자동으로 점검합니다."""
    check = evaluate_row({"structure": label, "case_id": case["id"], "answer": " ".join(collected.values())}, gold)
    return {"경로": label, "사례": case["id"], "수집 칸": list(collected), "자료 바이트": len(" ".join(collected.values()).encode()), "누락 사실": check["누락 사실"], "금지 포함": check["금지 포함"]}

# 복합 사례(T-1001)를 Fast Path로 강제 실행했을 때와 Slow Path로 실행했을 때의 수집 자료 비교
show_table([check_collected("Fast 강제", T1001, collect_fast(T1001)), check_collected("Slow", T1001, collect_slow(T1001))])
print("수집한 자료 자체를 검사한 결과이며 최종 답변을 검사한 것이 아닙니다. / 모델 호출 0회")

In [ ]:
# Fast Path를 먼저 실행한 뒤 부족한 자료 종류가 있으면 Slow Path로 넘어가는 복구 간선 그래프를 구성합니다.
class FallbackState(TypedDict, total=False):
    case: dict
    collected: dict
    lacking: list[str]

def fast_node(state):
    """먼저 비용이 낮은 Fast Path 수집을 실행합니다."""
    return {"collected": collect_fast(state["case"])}

def check_kinds(state):
    """질문에 필요한 자료 종류 중 아직 수집되지 않은 항목(lacking)이 있는지 점검합니다."""
    return {"lacking": [kind for kind in needed_kinds(state["case"]["question"]) if kind not in state["collected"]]}

def slow_node(state):
    """부족한 자료가 있을 때만 Slow Path 수집을 실행합니다."""
    return {"collected": collect_slow(state["case"])}

# Fast -> check_kinds -> (부족하면 slow, 충분하면 END) 그래프 컴파일
builder = StateGraph(FallbackState)
builder.add_node("fast", fast_node)
builder.add_node("check_kinds", check_kinds)
builder.add_node("slow", slow_node)
builder.add_edge(START, "fast")
builder.add_edge("fast", "check_kinds")
builder.add_conditional_edges("check_kinds", lambda state: "slow" if state["lacking"] else END, ["slow", END])
builder.add_edge("slow", END)
fallback_app = builder.compile()

# 단순 요청과 복합 요청을 넣었을 때의 실제 이동 경로 비교
rows = []
for case_id in ["simple-deadline", "complex-double-pay"]:
    path, state = walk(fallback_app, {"case": CASES[case_id]})
    rows.append({"경로 기록": " → ".join(path), "부족한 자료 종류": state["lacking"], **check_collected("복구 간선", CASES[case_id], state["collected"])})
show_table(rows, ["사례", "경로 기록", "부족한 자료 종류", "자료 바이트", "누락 사실"])
print("복구 판단은 수집된 자료 종류만 확인하며 평가 파일을 읽지 않습니다. / 모델 호출 0회")

### 그림으로 보기: 분기를 어디에 두는가

4절에서 만든 두 그래프를 나란히 비교해 보면 조건부 분기가 놓인 위치가 다릅니다. 위쪽은 사전 분류 방식인 `rule_app`이고, 아래쪽은 사후 복구 방식인 `fallback_app`입니다.

![분기의 두 자리. 위는 classify가 조건으로 경로를 먼저 고르는 규칙 라우터이고, 아래는 fast를 먼저 돌린 뒤 check_kinds가 자료 종류가 부족하면 slow로 보내는 복구 간선이다](./assets/03b-1-branch-placement.png)

- **사전 분기(위쪽)**: 작업을 시작하기 전에 `classify` 노드가 조건을 평가하여 경로를 먼저 선택합니다. 만약 경로를 잘못 선택하면 해당 경로의 불필요한 비용이나 핵심 사실 누락을 그대로 감수해야 합니다.
- **복구 간선(아래쪽)**: 일단 비용이 낮은 `fast`를 먼저 실행한 뒤, `check_kinds` 노드가 수집된 자료 종류를 점검하여 부족할 때만 `slow`로 보냅니다. 역방향 순환 없이 앞으로만 진행하지만, `slow`로 넘어갈 경우 앞서 `fast`에서 수행한 검색 비용은 매몰됩니다.

**관찰할 것.** 오분류의 비용은 단순한 정확도 수치 하나로 표현되지 않으며, 어느 방향으로 틀렸는지에 따라 전혀 다른 비용이 발생합니다.

- Router 비교표에서 `일치`가 `False`인 행의 오분류 방향을 살펴보세요. 기대 경로가 `slow`인데 `fast`로 분류했다면 필수 사실이 누락될 위험이 생기고, 반대로 기대 경로가 `fast`인데 `slow`로 분류했다면 답변은 정확하더라도 불필요한 모델 호출과 지연 시간을 더 쓰게 됩니다. 만약 6개 사례가 모두 일치했다면 이번 6개 표본과 1회 실행에서 오분류가 드러나지 않은 것뿐입니다. 표본 수가 적고 단 1회 관찰이므로 이것만으로 정확도가 완벽하다고 일반화할 수는 없습니다.
- `규칙 경로` 열과 `기대` 열을 비교해 보세요. 키워드 규칙은 실행 결과가 항상 일정하고 모델 호출 비용이 들지 않지만, 키워드가 겹치거나 빠져 있으면 곧바로 오분류합니다. 예를 들어 `complex-ops-log`는 두 문서를 대조해야 하는 복합 요청이지만, 키워드 규칙에서는 자료 종류가 하나로만 잡혀 Fast Path로 잘못 이동합니다(핵심 사실 누락 방향의 오류). 반대로 `simple-duplicate-rule`은 단순 규정 질문이지만 키워드가 두 종류 잡혀 Slow Path로 이동합니다(비용 증가 방향의 오류). 반면 모델 Router는 호출 비용이 발생하고 실행할 때마다 결과가 달라질 가능성이 있습니다.
- 실패 재현 표의 `누락 사실` 열은 복합 요청(`T-1001`)을 Fast Path로 강제 실행했을 때 어떤 핵심 정보들을 놓치게 되는지 보여 줍니다. 이때 코드 예외나 에러는 발생하지 않고 조용히 정보가 빠집니다. 즉, 복합 요청을 단순 경로로 보내면 **정보 누락**이 발생하고, 단순 요청을 복합 경로로 보내면 **과잉 비용**이 발생합니다.
- 복구 간선 표에서 단순 요청인 `simple-deadline`은 `fast`에서 바로 종료되고, 복합 요청인 `complex-double-pay`는 `check_kinds`를 거쳐 `slow`까지 이동합니다. 다만 여기서도 복구 여부를 판단하는 `check_kinds`가 키워드 규칙(`needed_kinds`)을 사용하므로, 키워드 판정이 틀리면 복구 간선도 함께 틀릴 수 있다는 점을 기억해 두세요.

복구 간선은 역방향 없이 앞으로만 이동합니다. 그렇다면 `slow` 경로를 실행한 뒤에도 여전히 정보가 부족할 때는 어떻게 해야 할까요? 부족한 부분만 다시 시도한다면 언제 멈추어야 할까요? 바로 다음 절에서 다룰 주제입니다.

## 5절. 루프와 종료 조건

**문제.** 생성된 초안이 검증을 통과하지 못했을 때 부족한 부분을 보완하여 다시 시도하는 구조를 **루프(Loop)** 라고 합니다. 흐름 자체는 '생성 → 검증 → 재시도 또는 종료'로 단순하지만, 실무에서 가장 까다로운 부분은 **언제 루프를 멈출 것인가(종료 조건)** 입니다.

**먼저 예측하세요.** 검증이 영원히 통과하지 못하는 루프에 별도의 종료 조건을 넣지 않으면 어떻게 될까요? 무한히 반복될까요, 아니면 프레임워크가 강제로 멈춰 줄까요? 만약 프레임워크가 멈춘다면 어떤 이름의 예외가 발생하며 어느 단계에서 멈출까요? 그리고 한 가지 더 생각해 보세요. 재시도 횟수 상한만 3회로 걸어 두었을 때, 세 번의 시도에서 매번 똑같은 항목이 누락된다면 이 루프가 세 번 동안 한 일은 결국 무엇일까요?

아래 코드에서는 먼저 종료 조건이 없는 루프를 실행하여 LangGraph가 어떻게 무한 루프를 차단하는지 확인하고, 이어서 3가지 종료 상태를 가진 재시도 루프를 시뮬레이션합니다.

In [ ]:
# 실패 재현: 종료 조건 없는 루프를 recursion_limit로 멈춥니다
class EndlessState(TypedDict, total=False):
    attempts: int

def try_once(state):
    """호출될 때마다 시도 횟수(attempts)만 1씩 증가시키는 노드입니다."""
    return {"attempts": state["attempts"] + 1}

# 무조건 자기 자신(try_once)으로 되돌아가는 무한 루프 그래프 구성
builder = StateGraph(EndlessState)
builder.add_node("try_once", try_once)
builder.add_edge(START, "try_once")
builder.add_conditional_edges("try_once", lambda state: "try_once", ["try_once", END])
endless_app = builder.compile()

# recursion_limit=5를 설정하여 프레임워크가 발생시키는 예외 확인
try:
    endless_app.invoke({"attempts": 0}, config={"recursion_limit": 5})
except GraphRecursionError as error:
    print(f"{type(error).__name__}: {str(error).splitlines()[0]}")
print("flows_of:", flows_of(endless_app))
print("모델 호출 0회")

In [ ]:
# 3가지 종료 조건(complete, same_failure, retry_limit)을 갖춘 재시도 루프 시뮬레이션
class RetryState(TypedDict, total=False):
    attempt: int
    history: Annotated[list, operator.add]
    stop: str

def build_retry_loop(find, required: set[str], max_attempts: int, same_limit: int):
    """find(n)은 n번째 시도에서 확보한 항목 집합입니다. same_limit번 연속으로 동일한 누락이 반복되면 즉시 멈춥니다."""
    def attempt(state):
        number = state.get("attempt", 0) + 1
        return {"attempt": number, "history": [sorted(required - find(number))]}

    def check(state):
        history, last = state["history"], state["history"][-1]
        same = len(history) >= same_limit and all(item == last for item in history[-same_limit:])
        stop = "complete" if not last else "same_failure" if same else "retry_limit" if state["attempt"] >= max_attempts else ""
        return {"stop": stop}

    builder = StateGraph(RetryState)
    builder.add_node("attempt", attempt)
    builder.add_node("check", check)
    builder.add_edge(START, "attempt")
    builder.add_edge("attempt", "check")
    builder.add_conditional_edges("check", lambda state: END if state["stop"] else "attempt", ["attempt", END])
    return builder.compile()

# 3가지 시나리오(점점 나아짐, 똑같은 누락 반복, 매번 다른 누락 발생) 정의 및 실행
REQUIRED = {"기한", "비율", "기준"}
SCENARIOS = {"나아짐": {1: {"기한"}, 2: {"기한", "비율"}, 3: REQUIRED}, "그대로": {n: {"기한"} for n in range(1, 9)},
             "매번 다름": {n: [{"기한"}, {"비율"}, {"기준"}][n % 3] for n in range(1, 9)}}

def run_scenarios(max_attempts: int, same_limit: int) -> list[dict]:
    rows = []
    for name, found in SCENARIOS.items():
        state = build_retry_loop(lambda n, f=found: f.get(n, set()), REQUIRED, max_attempts, same_limit).invoke({})
        rows.append({"시나리오": name, "종료 상태": state["stop"], "시도": state["attempt"], "시도별 누락": state["history"]})
    return rows

show_table(run_scenarios(max_attempts=5, same_limit=2))
print("모델 호출 0회")

### 그림으로 보기: 루프가 멈추는 길

위 표의 `종료 상태`는 `check` 노드가 아래 다이어그램의 순서대로 조건을 판정한 결과입니다. 위에서부터 차례로 검사하여 해당되는 조건이 있으면 루프를 종료하고, 어느 조건에도 해당하지 않을 때만 `attempt` 노드로 되돌아갑니다.

![재시도 루프의 검사 순서. 누락이 없으면 complete, 같은 누락이 반복되면 same_failure, 시도 수가 상한이면 retry_limit으로 멈추고 그 밖에는 attempt로 되돌아간다](./assets/03b-2-loop-exits.png)

- **`complete`가 최우선입니다**: 누락된 항목이 하나도 없다면 나머지 조건은 확인할 필요 없이 정상 종료합니다.
- **`same_failure`는 `retry_limit`보다 먼저 검사합니다**: 똑같은 누락이 연속으로 반복된다면 최대 시도 상한에 도달하기 전에 조기 중단하여 불필요한 비용 낭비를 막습니다.
- 최대 시도 횟수와 동일 실패 연속 허용 횟수는 각각 `max_attempts`와 `same_limit` 파라미터로 조절합니다. 뒷부분의 '직접 해보기 1'에서 이 두 값을 직접 변경해 봅니다.

위 시뮬레이션 표의 종료 상태는 세 가지로 구분됩니다. `complete`는 모든 필수 항목 확보 완료, `same_failure`는 동일한 실패의 연속 반복으로 인한 조기 중단, `retry_limit`은 최대 시도 횟수 도달입니다. 운영 환경에서는 이 세 가지 종료 상태에 따라 후속 조치가 달라져야 합니다. 특히 `same_failure`는 같은 방식으로 더 재시도해 보았자 결과가 달라지지 않는다는 명확한 신호입니다.

이제 동일한 루프 구조를 실제 모델과 검색 도구로 연결하여 실행해 보겠습니다. 초안에서 필수 항목이 채워졌는지는 **코드가** 검사하고, 누락된 항목에 대해서만 추가 검색을 수행한 뒤 초안을 다시 작성합니다. 모델에게 "빠진 내용이 있는지 스스로 검토해 보라"고 맡기지 않습니다. 대신 각 항목의 이름(`item`), 값(`value`), 근거 ID(`evidence_id`)를 구조화된 출력으로 받아, 실제로 이번 실행의 작업대(`Desk`)에 등록된 근거를 인용했는지 코드로 대조합니다. 원문에 해당 값이 없으면 빈 문자열로 비워 두도록 지시합니다.

또한 비교를 위해 코퍼스 원문에 아예 답이 존재하지 않는 항목(`법인카드 포인트 보상`)을 하나 추가하여 같은 루프를 한 번 더 실행합니다. 조건을 하나만 바꾸었을 때 루프가 어떤 종료 상태로 멈추는지 비교해 보세요. 아래 코드는 먼저 실제 모델을 사용하는 재시도 루프 그래프(`item_app`)를 구성하고 시각화합니다.

In [ ]:
# 초안의 개별 항목과 전체 응답을 담을 구조화된 출력 스키마 정의
class ItemFill(BaseModel):
    item: str
    value: str = ""
    evidence_id: str = ""

class ItemDraft(BaseModel):
    text: str
    items: list[ItemFill]

# 재시도 루프의 상태 스키마 정의
class ItemLoopState(TypedDict, total=False):
    question: str
    items: dict
    missing: list[str]
    attempts: Annotated[list, operator.add]
    stop: str

MAX_ATTEMPTS = 3
ITEMS = {"크레딧 비율": "월간 가용률 구간 서비스 크레딧 비율 기본 이용료", "계산 기준": "서비스 크레딧 계산 기준 사용량 초과 요금", "신청 기한": "서비스 크레딧 신청 기한"}
draft_model = model.with_structured_output(ItemDraft, method="json_schema")

def draft_node(state, config: RunnableConfig):
    """현재까지 확보된 근거만으로 초안을 작성하고, 필수 항목의 값과 근거 ID가 유효한지 코드로 검사합니다."""
    desk, meter = config["configurable"]["desk"], ModelMeter()
    evidence = [{"evidence_id": e.evidence_id, "quote": e.quote} for e in desk.context.evidence.values()]
    prompt = (f"근거만 써서 답합니다. 필수 항목: {list(state['items'])}. items에 항목 이름을 그대로 쓰고, 원문에 직접 적힌 값과 그 evidence_id를 쓰세요. 원문에 값이 없으면 value와 evidence_id를 비우세요.\n"
              f"{scope_line(desk)}\n질문: {state['question']}\n근거: {json.dumps(evidence, ensure_ascii=False)}")
    reply = draft_model.invoke(prompt, config={"callbacks": [meter]})
    known = desk.known_refs()
    missing = [name for name in state["items"] if not any(f.item == name and f.value and f.evidence_id in known for f in reply.items)]
    totals = meter.totals()
    record = {"시도": len(state.get("attempts", [])) + 1, "근거 수": len(evidence), "누락 항목": missing, "모델 호출": totals["model_calls"], "입력 토큰": totals["input_tokens"], "출력 토큰": totals["output_tokens"], "초안": reply.text}
    return {"missing": missing, "attempts": [record]}

def item_check(state):
    """누락 항목이 없으면 complete, 직전 시도와 누락 항목이 같으면 same_failure, 상한 도달 시 retry_limit을 판정합니다."""
    history = [a["누락 항목"] for a in state["attempts"]]
    same = len(history) >= 2 and history[-1] == history[-2]
    return {"stop": "complete" if not history[-1] else "same_failure" if same else "retry_limit" if len(history) >= MAX_ATTEMPTS else ""}

def research_node(state, config: RunnableConfig):
    """누락된 항목(missing)에 대해서만 추가로 문서를 검색하여 작업대(Desk)에 근거를 보충합니다."""
    desk = config["configurable"]["desk"]
    try:
        for name in state["missing"]:  # 빠진 항목만 다시 검색합니다.
            desk.context.search(state["items"][name], limit=2)
    except BudgetExceeded:
        return {"stop": "budget_exceeded"}
    return {}

# draft -> check -> (미완료 시 research -> draft) 루프 그래프 컴파일 및 시각화
builder = StateGraph(ItemLoopState)
for name, node in [("draft", draft_node), ("check", item_check), ("research", research_node)]:
    builder.add_node(name, node)
builder.add_edge(START, "draft")
builder.add_edge("draft", "check")
builder.add_conditional_edges("check", lambda state: END if state["stop"] else "research", ["research", END])
builder.add_conditional_edges("research", lambda state: END if state.get("stop") else "draft", ["draft", END])
item_app = builder.compile()
show_graph(item_app)

In [ ]:
# 기본 항목 세트와 코퍼스에 답이 없는 항목을 추가한 세트로 루프를 각각 실행합니다.
announce("초안 작성 → 필수 항목 검증 → 누락 항목 재검색 → 재작성 루프를 두 번 실행합니다", model_calls=6)

def run_item_loop(label: str, items: dict) -> dict:
    """초기 근거를 1건만 좁게 검색해 둔 상태에서 재시도 루프를 실행합니다."""
    desk = make_desk(T1001, label=label)
    desk.context.search("서비스 크레딧 비율", limit=1)  # 첫 시도에서 일부 항목이 빠지도록 초기 근거를 좁게 설정합니다.
    state = item_app.invoke({"question": "서비스 크레딧의 비율과 계산 기준, 신청 기한을 알려 주세요.", "items": items}, config={"configurable": {"desk": desk}})
    return {"label": label, "state": state, "desk": desk}

normal_loop = run_item_loop("기본", ITEMS)
no_answer_loop = run_item_loop("답 없는 항목 추가", {**ITEMS, "법인카드 포인트 보상": "법인카드 포인트 보상"})
for loop in (normal_loop, no_answer_loop):
    print(f"[{loop['label']}] 종료 상태: {loop['state']['stop']}, 총 시도: {len(loop['state']['attempts'])}회, 검색 호출: {loop['desk'].budget.counts.get('search', 0)}회")

In [ ]:
# 두 루프 실행의 시도별 기록을 표로 출력하고 마지막 초안을 확인합니다.
show_table([{"실행": loop["label"], **{k: v for k, v in attempt.items() if k != "초안"}} for loop in (normal_loop, no_answer_loop) for attempt in loop["state"]["attempts"]])
print("[기본] 마지막 초안:", normal_loop["state"]["attempts"][-1]["초안"])
print("[답 없는 항목 추가] 마지막 초안:", no_answer_loop["state"]["attempts"][-1]["초안"])

**관찰할 것.** 시도별 기록 표를 위에서 아래 순서로 따라가며 확인해 보세요.

- 시도 횟수가 거듭될수록 `누락 항목` 열의 개수가 줄어드는지 확인합니다. 줄어들었다면 `research` 노드의 추가 검색이 실제로 효과를 낸 것입니다. 이와 함께 `근거 수`가 늘어난 만큼 다음 시도의 `입력 토큰`도 함께 증가했는지 비교해 보세요. 재시도는 결코 공짜가 아닙니다.
- 만약 어떤 시도의 `근거 수`가 직전 시도와 똑같다면, 추가 검색을 수행했음에도 새로운 근거 문서를 하나도 찾지 못했다는 뜻입니다. 그 상태에서 다시 `draft` 노드를 호출하는 것은 사실상 직전과 똑같은 프롬프트를 모델에 다시 보내는 셈입니다. 그렇다면 모델을 다시 호출하기 전에 조기 중단하는 규칙을 어디에 넣을 수 있을까요? `research` 노드가 검색 전후의 무엇을 비교하면 될지 생각해 보세요.
- 첫 번째 시도부터 `누락 항목`이 비어 있었다면 모델이 좁은 초기 근거(1건) 안에서 세 항목의 값을 모두 찾아 채운 것입니다. 이때는 채워 넣은 값이 원문에 실제로 적혀 있는지 마지막 초안과 대조해 보세요. 코드의 `missing` 검사는 `evidence_id`가 작업대에 등록되어 있는지만 확인할 뿐, 값의 의미가 원문과 일치하는지는 판단하지 못합니다(이 한계는 다음 `04a` 노트북에서 자세히 다룹니다).
- `답 없는 항목 추가` 실행이 어떤 `종료 상태`로 멈추었는지 확인해 보세요. `same_failure`로 종료되었다면 그래프가 '동일한 누락의 반복'을 감지하고 상한(3회)에 도달하기 전에 스스로 멈춘 것입니다. 마지막 초안에서 해당 항목(`법인카드 포인트 보상`)을 어떻게 서술했는지도 읽어 보세요. 반대로 만약 `complete`로 끝났다면 모델이 관련 없는 근거 ID를 붙여 값을 억지로 채워 넣었을 가능성이 있습니다.
- 두 실행(`기본` vs `답 없는 항목 추가`)의 1차 시도를 서로 비교해 보세요. 초기 근거는 똑같이 1건인데 `누락 항목` 목록이 다르게 나왔다면 모델이 빈칸 여부를 실행마다 조금씩 다르게 판단한 것입니다. 만약 어떤 항목이 `누락 항목`에서 빠졌는데도 정작 마지막 초안에는 구체적인 수치 대신 `"관련 약정을 따른다"` 같은 포괄적 문장만 적혀 있다면, 모델이 추상적인 문구를 `value`에 채워 넣었고 코드의 단순 빈칸 검사(`bool(f.value)`)가 이를 통과시킨 사례입니다. 검사가 느슨한 만큼 재검색 루프도 일찍 종료됩니다.
- 앞서 살펴본 시뮬레이션 표와 연결해 보세요. `same_failure`와 `retry_limit`은 루프가 멈춘 이유가 다릅니다. 똑같은 프롬프트와 똑같은 근거로 모델을 다시 호출하는 것은 재시도 전략이 아닙니다. 재시도 루프에서 매 시도마다 달라져야 하는 핵심은 프롬프트에 들어가는 **근거(Evidence)** 여야 합니다.

지금까지 네 가지 기본 흐름(순차·병렬·분기·루프)을 모두 직접 구현해 보았습니다. 다음 6절에서는 다양한 멀티에이전트 패턴 이름들을 이 네 가지 흐름으로 번역합니다.

## 6절. 패턴을 네 흐름으로 번역

**문제.** 업계에서는 새로운 에이전트 패턴 이름이 계속해서 등장합니다. 패턴 이름마다 별개의 기술처럼 새로 배우면 끝이 없고, 이름이 다르다는 이유만으로 전혀 다른 실행 구조라고 믿으면 객관적인 비교를 할 수 없습니다. 이번 절에서는 자주 언급되는 다섯 가지 대표 패턴을 우리가 배운 **네 가지 기본 흐름(순차·병렬·분기·루프)** 으로 번역합니다. 이 번역이 익숙해지면 패턴 이름에 얽매이지 않고 *"순서는 어떠한가? 무엇이 독립적인가? 분기 조건은 무엇인가? 언제 종료되는가?"* 라는 네 가지 질문만으로 어떤 아키텍처든 분석하고 비교할 수 있습니다.

| 패턴 | 순차 | 병렬 | 분기 | 루프 | 한 줄 번역 |
|---|---|---|---|---|---|
| Router | 분류 후 처리 | 없음 | 있음 | 없음 | 조건부 간선 하나입니다 |
| Supervisor | 지시 후 보고 | 필요시 사용 | 있음 | 있음 | 분기와 루프의 결합입니다 |
| Orchestrator-Worker | 분해 → 집계 | 있음 | 검증 후 분기 | 재계획 루프 | 분해 → 병렬 실행 → 집계입니다 |
| Handoff | 담당자(owner) 전환 | 없음 | 있음 | 왕복 전환 | 담당자(owner) 전환과 반복입니다 |
| Hierarchical | 상위·하위 층 사이 | 하위 팀 내부 | 각 층마다 | 각 층마다 | 위 구조들의 다층 중첩입니다 |

**먼저 예측하세요.** 아래 코드에서는 일반 Python 함수 노드로 다섯 패턴의 뼈대 그래프를 만들고 `flows_of` 함수로 정적 구조를 분석합니다. `flows_of`는 그래프에 선언된 노드와 간선만 읽어서 조건부 간선 수와 순환(Cycle) 유무를 계산합니다. 여기서 `Supervisor`는 하위 Worker를 한 명씩 차례로 호출하고, `Hierarchical`은 하위 팀 내부에서 `Send`를 사용해 팀원 두 명을 동시에 병렬로 실행합니다. 과연 두 뼈대 그래프의 `flows_of` 정적 분석 결과는 얼마나 다를까요? 정적 간선 정보만 보고 어느 쪽이 병렬 실행을 포함하는지 구별할 수 있을까요?

아래 코드에서는 먼저 Router, Supervisor, Orchestrator-Worker 세 가지 패턴의 뼈대 그래프를 생성합니다.

In [ ]:
# 다섯 패턴의 구조적 뼈대를 비교하기 위한 최소 상태 스키마와 헬퍼 함수 정의
class SkeletonState(TypedDict, total=False):
    log: Annotated[list, operator.add]
    turns: int

def step(name: str, seconds: float = 0.05):
    """실행 구간(span)을 기록하고 잠시 대기하는 일반 함수 노드를 생성합니다."""
    def node(state, config: RunnableConfig):
        with config["configurable"]["trace"].span(name):
            time.sleep(seconds)
        return {"log": [name], "turns": state.get("turns", 0) + 1}
    return node

def member(payload: dict, config: RunnableConfig):
    """Send를 통해 병렬로 호출되는 하위 작업자 노드입니다."""
    with config["configurable"]["trace"].span(payload["task_id"]):
        time.sleep(0.05)
    return {"log": [payload["task_id"]]}

def graph_of(nodes: dict, edges: list, conditional: list):
    """노드, 일반 간선, 조건부 간선 목록을 받아 StateGraph를 컴파일합니다."""
    builder = StateGraph(SkeletonState)
    for name, node in nodes.items():
        builder.add_node(name, node)
    for start, end in edges:
        builder.add_edge(start, end)
    for start, chooser, targets in conditional:
        builder.add_conditional_edges(start, chooser, targets)
    return builder.compile()

# 1) Router, 2) Supervisor, 3) Orchestrator-Worker 뼈대 그래프 조립
fan_two = lambda state: [Send("member", {"task_id": f"member_{n}"}) for n in ("a", "b")]
router_skeleton = graph_of({"route": step("route"), "fast": step("fast"), "slow": step("slow")}, [(START, "route"), ("fast", END), ("slow", END)],
                           [("route", lambda s: "slow", ["fast", "slow"])])
supervisor_skeleton = graph_of({"supervisor": step("supervisor"), "worker_a": step("worker_a"), "worker_b": step("worker_b")},
                               [(START, "supervisor"), ("worker_a", "supervisor"), ("worker_b", "supervisor")],
                               [("supervisor", lambda s: {1: "worker_a", 3: "worker_b"}.get(s["turns"], END), ["worker_a", "worker_b", END])])
orchestrator_skeleton = graph_of({"plan": step("plan"), "member": member, "aggregate": step("aggregate")},
                                 [(START, "plan"), ("member", "aggregate"), ("aggregate", END)], [("plan", fan_two, ["member"])])
print("뼈대 3개 노드 수:", [flows_of(app)["노드"] for app in (router_skeleton, supervisor_skeleton, orchestrator_skeleton)], "개 노드")

In [ ]:
# 4) Handoff(Command 기반 담당자 전환)와 5) Hierarchical(하위 그래프 중첩) 뼈대를 추가하고 다섯 패턴을 비교합니다.
def hand_over(name: str, other: str, state: dict, config: RunnableConfig) -> Command:
    """현재 담당자(owner)가 작업을 수행한 뒤 Command(goto=...)로 다음 담당자에게 제어권을 넘깁니다."""
    with config["configurable"]["trace"].span(name):
        time.sleep(0.05)
    turns = state.get("turns", 0) + 1
    return Command(update={"log": [name], "turns": turns}, goto=END if turns >= 3 else other)

def owner_a(state, config: RunnableConfig) -> Command[Literal["owner_b", "__end__"]]:
    return hand_over("owner_a", "owner_b", state, config)

def owner_b(state, config: RunnableConfig) -> Command[Literal["owner_a", "__end__"]]:
    return hand_over("owner_b", "owner_a", state, config)

def team(name: str):
    """내부에서 lead -> Send(member_a, member_b) 병렬 실행을 수행하는 하위 팀 그래프를 노드로 감쌉니다."""
    inner = graph_of({"lead": step(f"{name}_lead"), "member": member}, [(START, "lead"), ("member", END)], [("lead", fan_two, ["member"])])
    def node(state, config: RunnableConfig):  # 부모 그래프의 log가 중복 합산되지 않도록 빈 리스트로 하위 그래프를 호출합니다.
        return {"log": inner.invoke({"log": []}, config=config)["log"], "turns": state.get("turns", 0) + 1}
    return node

handoff_skeleton = graph_of({"owner_a": owner_a, "owner_b": owner_b}, [(START, "owner_a")], [])
hierarchical_skeleton = graph_of({"boss": step("boss"), "team_a": team("team_a"), "team_b": team("team_b")},
                                 [(START, "boss"), ("team_a", "boss"), ("team_b", "boss")],
                                 [("boss", lambda s: {1: "team_a", 3: "team_b"}.get(s["turns"], END), ["team_a", "team_b", END])])
SKELETONS = {"Router": router_skeleton, "Supervisor": supervisor_skeleton, "Orchestrator-Worker": orchestrator_skeleton,
             "Handoff": handoff_skeleton, "Hierarchical": hierarchical_skeleton}
flow_rows = [{"패턴": name, **flows_of(app)} for name, app in SKELETONS.items()]
show_table(flow_rows)
print("모델 호출 0회")

In [ ]:
# 다섯 패턴 뼈대를 실제로 실행하여 시간 겹침(병렬 여부)과 실제 실행 순서를 측정합니다.
rows = []
for flow_row in flow_rows:
    trace = Trace(flow_row["패턴"])
    state = SKELETONS[flow_row["패턴"]].invoke({"log": []}, config={"configurable": {"trace": trace}})
    rows.append({**flow_row, "겹침(초)": overlap_seconds(trace.spans), "병렬": overlap_seconds(trace.spans) > 0, "실행 순서": " → ".join(state["log"])})
show_table(rows, ["패턴", "조건부 간선", "순환", "겹침(초)", "병렬", "실행 순서"])
print("모델 호출 0회")

위 표에서 `조건부 간선`과 `순환` 열은 컴파일된 그래프의 구조를 **정적으로 분석**하여 얻은 값이고, `겹침(초)`와 `병렬` 열은 그래프를 **실제로 실행한 추적 기록(Trace)** 에서 얻은 값입니다. `Supervisor`와 `Hierarchical` 두 행을 나란히 비교해 보세요. 앞선 정적 구조 열(`조건부 간선`, `순환`)은 똑같은데 실제 실행 결과인 `병렬` 열만 다릅니다. `Hierarchical`의 하위 팀 내부에서 일어나는 병렬 실행(`Send`)은 상위 그래프의 간선 구조에 드러나지 않기 때문입니다. 또한 `Orchestrator-Worker`처럼 상위 그래프에 `Send`가 직접 있더라도 정적 그래프 상에서는 조건부 간선 하나로만 표시되므로, 경로 하나를 고르는 분기인지 여러 Worker를 동시에 띄우는 병렬인지는 간선 모양만으로 구별할 수 없습니다. 즉, **정적 구조 분석과 실행 Trace 측정은 서로를 대체할 수 없으며 함께 확인해야 합니다.**

이제 다섯 패턴 중 하나인 **Orchestrator-Worker**를 골라 네 가지 흐름이 모두 들어가도록 끝까지 조립해 보겠습니다. 전체 흐름을 `plan → Send(worker) → aggregate → validate → replan`으로 연결하고 각 Node가 네 흐름 중 어떤 역할을 맡는지 표시합니다. 먼저 모델을 부르지 않는 일반 함수(`plain_work`)로 실행해 동작을 검증합니다. 이때 `outage` 작업이 1차 시도에서 실패하도록 의도적으로 설정해 두었습니다. `replan` 노드는 실패한 작업만 골라 `outage#2`처럼 새로운 작업 키를 부여하여 다시 `Send`로 보냅니다. 앞서 `03a`에서 살펴본 것처럼 `merge_results` 리듀서는 동일한 키에 서로 다른 결과가 덮어써지는 것을 거부하므로, 재시도 결과는 새 키로 기록하고 합류 함수인 `join_results`가 각 작업의 가장 마지막 시도 결과를 읽어 오도록 설계되어 있습니다.

### 그림으로 보기: Orchestrator-Worker에서 네 흐름이 있는 자리

위에서 설명한 조립 구조가 바로 아래 코드 셀의 `build_orchestrator` 함수가 생성하는 그래프입니다. 코드 안의 `FLOW_OF_NODE` 딕셔너리가 각 Node에 붙인 흐름 설명을 다이어그램 위에 함께 표시했습니다.

![Orchestrator-Worker 조립 그래프. plan, worker, aggregate는 순차이고 worker는 Send로 병렬이며 validate는 분기이고 replan에서 worker로 되돌아가는 간선이 루프다](./assets/03b-3-orchestrator-flows.png)

- **순차와 병렬**: `plan → worker → aggregate`는 선행 조건을 따르는 순차 흐름이고, 그 안의 `worker` 노드는 `Send`를 통해 여러 작업이 동일한 단계에서 병렬로 실행됩니다.
- **분기와 루프**: `validate` 노드 뒤의 조건부 간선이 분기 흐름이고, 검증 실패 시 `replan`을 거쳐 다시 `worker`로 되돌아가는 간선이 루프 흐름입니다.
- 만약 `replan` 단계에서 실패하거나 누락된 Worker가 없는데 최종 답변 검증(`validate_final`)만 실패한 경우라면, Worker를 다시 실행할 필요 없이 `aggregate`(작성자)로 바로 이동합니다(점선 경로). 이때는 기존 Worker 수집 결과를 그대로 두고 최종 답변만 다시 작성합니다.

아래 코드는 `build_orchestrator` 빌더 함수를 정의하고 각 노드가 담당하는 흐름을 표로 출력합니다.

In [ ]:
# Orchestrator-Worker 전체 흐름(순차·병렬·분기·루프)을 담는 상태 스키마와 그래프 빌더 정의
class OrchestratorState(TypedDict, total=False):
    tasks: list[dict]
    results: Annotated[dict, merge_results]
    final: dict
    problems: list[str]
    replans: int
    problem_log: Annotated[list, operator.add]
    order: Annotated[list, operator.add]

FLOW_OF_NODE = {"plan": "순차: 분해가 먼저입니다", "worker": "병렬: Send로 같은 단계에 실행합니다", "aggregate": "순차: 모두 끝나야 합칩니다",
                "validate": "분기: 통과, 재계획, 포기", "replan": "루프: 부족한 작업만 worker로 돌아갑니다"}

def build_orchestrator(tasks: list[dict], work, write, check, max_replans: int = 1):
    """작업 분배(plan) -> 병렬 실행(worker) -> 집계(aggregate) -> 검증(validate) -> 재계획(replan) 그래프를 조립합니다."""
    def send_tasks(state):
        return [Send("worker", {"task": t}) for t in state["tasks"]] or "aggregate"
    def worker(payload, config: RunnableConfig):
        return {"results": {payload["task"]["task_id"]: work(payload["task"], config)}, "order": [f"worker:{payload['task']['task_id']}"]}
    def replan(state, config: RunnableConfig):
        number = state.get("replans", 0) + 1
        info = join_results({k: WorkerResult.model_validate(v) for k, v in state["results"].items()}, [t["task_id"] for t in tasks])
        retry = [{**t, "task_id": f"{t['task_id']}#{number + 1}", "attempt": number + 1} for t in tasks if t["task_id"] in info["failed"] + info["missing"]]
        return {"tasks": retry, "replans": number, "order": ["replan"]}
    def validate(state, config: RunnableConfig):
        problems = check(state, config)
        return {"problems": problems, "problem_log": [problems], "order": ["validate"]}
    nodes = {"plan": lambda state: {"tasks": tasks, "order": ["plan"]}, "worker": worker, "replan": replan, "validate": validate,
             "aggregate": lambda state, config: {"final": write(state, config), "order": ["aggregate"]}}
    builder = StateGraph(OrchestratorState)
    for name, node in nodes.items():
        builder.add_node(name, node)
    builder.add_edge(START, "plan")
    builder.add_conditional_edges("plan", send_tasks, ["worker", "aggregate"])
    builder.add_edge("worker", "aggregate")
    builder.add_edge("aggregate", "validate")
    builder.add_conditional_edges("validate", lambda s: "replan" if s["problems"] and s.get("replans", 0) < max_replans else END, ["replan", END])
    builder.add_conditional_edges("replan", send_tasks, ["worker", "aggregate"])
    return builder.compile()

def worker_problems(state: dict, required: list[str]) -> list[str]:
    """필수 작업 중 누락되었거나 실패한 Worker 목록을 점검합니다."""
    info = join_results({k: WorkerResult.model_validate(v) for k, v in state["results"].items()}, required)
    return [f"{name}: 결과 없음" for name in info["missing"]] + [f"{name}: 실패" for name in info["failed"]]

TASK_IDS = [t["task_id"] for t in PLAN["tasks"]]
show_table([{"Node": node, "흐름": flow} for node, flow in FLOW_OF_NODE.items()])
print("입력 작업 목록은 앞선 PLAN과 동일합니다:", TASK_IDS)

In [ ]:
# 먼저 모델을 부르지 않는 일반 함수(plain_work)로 Orchestrator-Worker 그래프의 재계획 흐름을 검증합니다.
def plain_work(task: dict, config: RunnableConfig) -> dict:
    """outage 첫 시도(outage)만 실패를 반환하고, 나머지 작업과 재시도(outage#2)는 성공을 반환합니다."""
    with config["configurable"]["trace"].span(task["task_id"]):
        time.sleep(0.05)
    if task["task_id"] == "outage":
        return WorkerResult(task_id="outage", status="failed", error="주입한 첫 시도 실패").model_dump()
    return WorkerResult(task_id=task["task_id"], attempt=task.get("attempt", 1), status="complete", facts=[f"{task['task_id']} 확인"], evidence_refs=["F-예시"]).model_dump()

plain_app = build_orchestrator(PLAN["tasks"], plain_work, lambda state, config: {"작업 수": len(state["results"])},
                               lambda state, config: worker_problems(state, TASK_IDS))
trace = Trace("조립")
plain_state = plain_app.invoke({}, config={"configurable": {"trace": trace}})
print("실행 순서:", " → ".join(plain_state["order"]))
print("결과 키:", sorted(plain_state["results"]), "/ 재계획 횟수:", plain_state.get("replans", 0))
print("검증별 문제:", plain_state["problem_log"])
print(timeline(trace.spans))
show_graph(plain_app)
print("flows_of:", flows_of(plain_app), "/ 모델 호출 0회")

In [ ]:
# 동일한 build_orchestrator 구조에 실제 모델 Worker(live_work)와 작성자(live_write), 검증(live_check) 함수를 연결합니다.
def live_work(task: dict, config: RunnableConfig) -> dict:
    """실제 Deep Agent Worker를 실행하고 예산 초과 시 failed 상태의 WorkerResult로 변환합니다."""
    desk = config["configurable"]["desk"]
    try:
        return run_worker(WorkerTask.model_validate(task), desk, model).model_dump()
    except BudgetExceeded as error:
        return WorkerResult(task_id=task["task_id"], status="failed", error=str(error)).model_dump()

def live_write(state: dict, config: RunnableConfig) -> dict:
    """수집된 Worker 결과들을 바탕으로 최종 고객 답변(FinalAnswer)을 작성합니다."""
    desk = config["configurable"]["desk"]
    facts = {k: {f: v.get(f) for f in ("status", "facts", "evidence_refs", "uncertainty", "error")} for k, v in state["results"].items()}
    allowed = ROLE_TOOLS["writer"]
    agent = create_deep_agent(model=model, tools=desk.tools("writer"), system_prompt=f"고객에게 보낼 답을 씁니다. {scope_line(desk)}\n{OUTCOME_GUIDE}",
                              middleware=allowlist_middleware(desk.budget, allowed), permissions=DENY_WRITES, response_format=ProviderStrategy(FinalAnswer), name="writer")
    content = (f"고객 요청: {T1001['question']}\nWorker 결과: {facts}\n앞선 검증 문제: {state.get('problems') or '없음'}\n"
               "크레딧 금액은 calculate_credit으로 계산하고, 기본 이용료분만 계산 기준에 넣으세요. 실패하거나 누락된 작업은 uncertainty에 쓰세요.")
    with desk.trace.span("writer", "write"):
        reply = agent.invoke({"messages": [{"role": "user", "content": content}]}, config={"recursion_limit": 40})
    return reply["structured_response"].model_dump()

def live_check(state: dict, config: RunnableConfig) -> list[str]:
    """Worker 누락/실패 여부와 최종 답변의 근거·수치 정합성을 함께 검사합니다."""
    desk = config["configurable"]["desk"]
    return worker_problems(state, TASK_IDS) + validate_final(FinalAnswer.model_validate(state["final"]), desk.known_refs(), desk.source_text())

live_app = build_orchestrator(PLAN["tasks"], live_work, live_write, live_check)
print("앞 셀과 동일한 build_orchestrator이며, 교체한 것은 work, write, check 세 함수뿐입니다.")
print("flows_of:", flows_of(live_app))

In [ ]:
# 조립한 Orchestrator-Worker 그래프를 실제 모델 Worker로 관통 사례(T-1001)에 실행합니다.
announce("Orchestrator-Worker 조립 그래프를 실제 Worker로 T-1001에 실행합니다", model_calls=15)
live_desk, live_meter, started = make_desk(T1001, label="조립"), ModelMeter(), time.perf_counter()
live_state = live_app.invoke({}, config={"callbacks": [live_meter], "configurable": {"desk": live_desk}})
live_seconds = round(time.perf_counter() - started, 3)

# 최종 답변에 대해 승인 게이트(gate_answer)와 사후 자동 평가(evaluate_row) 적용
live_final = gate_answer(FinalAnswer.model_validate(live_state["final"]), live_desk, approved=None, request_id="flows-t1001")
cited = sorted({e.doc_id for ref, e in live_desk.context.evidence.items() if ref in live_final.evidence_refs})
live_check_row = evaluate_row({"structure": "조립", "case_id": T1001["id"], "answer": live_final.text, "uncertainty": live_final.uncertainty, "outcome": live_final.outcome,
                               "cited_docs": cited, "ledger_refunds": len(live_desk.ledger.refunds), "status": "ok"}, gold)
totals = live_meter.totals()

print("실행 순서:", " → ".join(live_state["order"]))
print("검증별 문제:", live_state["problem_log"])
print(timeline(live_desk.trace.spans))
show_table([{"전체(초)": live_seconds, "겹침(초)": overlap_seconds(live_desk.trace.spans), "모델 호출": totals["model_calls"], "입력 토큰": totals["input_tokens"], "재계획": live_state.get("replans", 0),
             "남은 문제": live_state["problems"], "outcome": live_final.outcome, "원장 건수": len(live_desk.ledger.refunds), "자동 검사": live_check_row["자동 검사"], "실패 칸": live_check_row["실패 칸"], "누락 사실": live_check_row["누락 사실"]}])
print(live_final.text)

**관찰할 것.** 네 가지 기본 흐름으로 분해해 보면 패턴 이름에 현혹되지 않고 실행 결과를 명확히 읽어 낼 수 있습니다. 위 실행 출력을 네 가지 흐름별로 하나씩 확인해 보세요.

- **순차**: `실행 순서`에서 `plan`이 맨 앞에 위치하고, 모든 Worker가 끝난 뒤에 `aggregate`가 실행됩니다. 타임라인(`timeline`)에서도 `writer` 구간은 세 Worker 구간이 모두 종료된 뒤에야 시작됩니다.
- **병렬**: 표의 `겹침(초)`가 `0`보다 큰지 확인합니다. 이 겹침은 그래프 코드(`Send`)가 강제한 병렬 실행의 결과이므로 모델의 판단과 무관하게 항상 발생합니다. 다만 개별 Worker 구간의 실행 시간은 각 모델이 도구를 몇 번 호출했는지에 따라 달라집니다.
- **분기와 루프**: 표의 `재계획` 열이 `0`이면 첫 번째 `validate`에서 모든 검사를 통과해 즉시 종료된 것입니다. 만약 `1`이라면 `검증별 문제`의 첫 번째 리스트가 재계획 루프를 촉발한 원인입니다. 이때 `실행 순서`에서 `replan` 바로 뒤에 `worker:`가 나타나면 실패한 Worker만 다시 실행한 것이고, `worker:` 없이 곧바로 `aggregate`가 나타나면 Worker 결과는 모두 정상이었으나 작성자의 최종 답변 검증(`validate_final`)에서 문제가 발견되어 답변만 다시 작성한 것입니다. 만약 마지막 `남은 문제` 열이 비어 있지 않다면 재계획 상한(`max_replans=1`)에 걸려 미해결 문제를 남긴 채 종료된 것이며, 이러한 미해결 문제를 고객 답변에 투명하게 드러내는 방법은 다음 노트북(`04a`, `04b`)에서 자세히 다룹니다.
- **승인 게이트와 자동 검사**: `outcome`이 `needs_approval`이고 `원장 건수`가 `0`이면 코드 게이트(`gate_answer`)가 실제 환불 실행을 안전하게 보류한 것입니다. 또한 `자동 검사`가 `False`라면 `실패 칸`과 `누락 사실`을 확인하고, 단순한 표현·표기 차이인지 실제로 핵심 내용이 빠진 것인지 출력된 답변 원문과 직접 대조해 보세요.

앞으로 새로운 멀티에이전트 패턴 이름을 만나면 *"순서는 무엇인가? 무엇이 서로 독립적인가? 어떤 조건으로 갈라지는가? 언제 반복을 멈추는가?"* 라는 네 가지 질문부터 던져 보세요. 이 네 질문에 답할 수 있다면 어떤 패턴이든 직접 구현하고 다른 구조와 공정하게 비교할 수 있습니다. 반대로 이 질문에 답하지 못하는 패턴 이름은 아직 구체적인 실행 구조를 정의하지 않은 마케팅 용어일 뿐입니다.

## 직접 해보기

수강생이 자신의 업무 시나리오나 기준에 맞추어 값을 직접 바꾸어 보는 실습 섹션입니다. 두 실습 모두 모델을 호출하지 않으므로 값을 수정한 뒤 그대로 셀을 실행하시면 됩니다.

### 해보기 1

5절의 재시도 시뮬레이션에서 최대 시도 상한(`MY_MAX_ATTEMPTS`)과 동일 실패 연속 판정 기준(`MY_SAME_LIMIT`)을 바꾸어 각 시나리오의 종료 상태를 비교해 보세요. `MY_SAME_LIMIT`을 `MY_MAX_ATTEMPTS`보다 크게 설정하면 `그대로` 시나리오는 어떤 상태로 끝나나요? 반대로 `MY_MAX_ATTEMPTS`를 `2`로 줄이면 `나아짐` 시나리오는 `complete`에 도달할 수 있나요? 자신의 업무에서는 어떤 설정값이 적절할지 확인해 보세요. 아래 코드는 수정한 파라미터로 세 가지 재시도 시나리오를 실행합니다.

In [ ]:
# 재시도 상한과 동일 실패 판정 횟수를 수정하며 종료 상태 변화를 확인합니다.
MY_MAX_ATTEMPTS = 5
MY_SAME_LIMIT = 4
show_table(run_scenarios(max_attempts=MY_MAX_ATTEMPTS, same_limit=MY_SAME_LIMIT))
print("모델 호출 0회")

### 해보기 2

앞선 `03a`의 '해보기 1'에서 작성했던 업무(`MY_TASKS`)나 본인의 실무 작업 하나를 골라, 네 가지 기본 흐름이 각각 어디에 해당하는지 정리해 보세요. 순차와 병렬은 작업 간의 **선행 조건**에서 찾고, 분기와 루프는 **중간 결과값에 따라 경로가 달라지는 지점**과 **검증 실패 시 다시 시도하는 지점**에서 찾습니다. 이 셀은 본인의 아키텍처 설계 기록이므로 정해진 정답이 없습니다. 마지막 `06` 노트북에서 이 설계 기록들을 한데 모아 다시 확인하므로 본인의 언어로 구체적으로 작성해 보세요. 아래 코드는 작성한 네 가지 흐름 설계를 로그 파일에 기록합니다.

In [ ]:
# 내 업무에서의 순차·병렬·분기·루프 흐름을 정의하고 설계 로그에 저장합니다.
MY_FLOWS = """순차: (내 업무에서 앞 작업이 끝나야 시작하는 것)
병렬: (서로 결과가 필요 없는 작업과, 동시에 돌리면 얻는 시간)
분기: (다음 작업을 가르는 조건과, 잘못 갈랐을 때의 비용)
루프: (다시 시도하는 조건과 멈추는 조건)"""
print(write_design_log("작업 Graph: 순차·병렬·분기·루프는?", MY_FLOWS).name, "파일에 기록했습니다.")
print(MY_FLOWS)

## 정리

| 흐름 | 그래프 도구 | 막아 주는 실패 | 재현한 실패 |
|---|---|---|---|
| 분기 | 조건부 간선 (`add_conditional_edges`) | 모든 요청에 동일한 고비용 지출 | 복합 요청을 Fast Path로 보냈을 때의 핵심 사실 누락 |
| 루프 | 순환 간선, 최대 시도 상한, 동일 실패 판정 | 보완 가능한 일시적 실패의 조기 포기 | 종료 조건 누락 시 `GraphRecursionError` 발생 |

- **패턴의 네 흐름 번역**:
  - **Router**는 조건부 간선 하나로 경로를 나누는 **분기**입니다.
  - **Supervisor**는 중앙에서 작업자를 선택하는 **분기**와 보고를 받고 반복하는 **루프**의 결합이며, 필요에 따라 **병렬**을 함께 사용합니다.
  - **Orchestrator-Worker**는 작업 분해 후 **병렬** 실행 및 집계로 이어지는 **순차** 흐름에, 검증 후 재계획하는 **분기·루프**를 결합한 구조입니다.
  - **Handoff**는 담당자(owner) 간 제어권 전환(**분기**)과 왕복(**루프**)입니다.
  - **Hierarchical**은 이러한 구조들을 상하 계층으로 중첩한 형태입니다.
- **정적 분석과 실행 Trace의 차이**: `flows_of`는 그래프의 정적 간선만 세기 때문에, 하위 그래프 내부의 병렬이나 `Send`를 통한 실제 동시 실행 여부는 반드시 실행 Trace의 `겹침(초)`로 확인해야 합니다.

## 확인 질문

1. Router의 성능을 단순 정확도 수치 하나로만 평가하면 어떤 비용의 차이를 놓치게 되나요?
2. Supervisor 패턴을 이 노트북에서 배운 네 가지 흐름으로 번역해 보세요. 정적 분석 함수인 `flows_of`가 구별해 주지 못하는 항목은 무엇인가요?
3. 최대 시도 상한(`max_attempts`)만 있고 동일 실패 연속 판정(`same_failure`)이 없는 루프는 어떤 불필요한 비용을 소모하나요? 의미 있는 재시도가 되려면 매 시도마다 무엇이 달라져야 하나요?

다음 `04a`, `04b` 노트북에서는 오늘 만든 네 가지 흐름을 실제 운영 환경에 투입할 수 있도록 다듬습니다. Worker 사이에 주고받는 데이터를 명시적인 **계약(Contract)** 으로 정의하고, 공유 **예산(budget)** 을 설정하며, 실패 유형별로 다음 행동을 세분화하고, 부분 실패 합류와 승인 게이트 및 멱등성을 그래프에 결합합니다.